# AI512 — Laboratory Sheet: Adversarial Search and Game Playing

**Course:** Artificial Intelligence (AI512)  
**Lab topic:** Minimax, Alpha–Beta Pruning, Nim, and Grundy's Game  
**Student Name:** Deepak Kumar  
**Roll Number:** 2026csm1012  
**Date:** October 24, 2026  

## Learning objectives
- evaluate a deterministic, two-player game tree using Minimax;
- implement Alpha–Beta pruning and count evaluated terminal states;
- model standard Nim and identify winning moves;
- model Grundy's Game and evaluate legal pile-splitting moves;
- explain the difference between standard Nim and Grundy's Game.


## Part 1 — Minimax Search

Minimax is used for deterministic, two-player, zero-sum games with perfect information. **MAX** chooses moves to maximize the utility, while **MIN** chooses moves to minimize it.

The tree has the structure `MAX → MIN → MAX → terminal utility`. The order of children is fixed in `GAME_TREE`; retain that order when evaluating the tree.

### Game-tree visual
```text
                         R (MAX)
                    /       |       \
                A (MIN)   B (MIN)   C (MIN)
                /   \     /   \     /   \
              A1    A2   B1    B2   C1    C2       (MAX)
             / \   / \  / \   / \  / \   / \
            3   5  6 9 1   2  0 -1 7  4 5   6  (terminal utilities)
```


### Task 1 — Implement and analyse Minimax

| Node | Minimax value |
|---|---:|
| A | 5 |
| B | 0 |
| C | 6 |
| R | 6 |

**Explanation & Justification:**  
- **Bottom-level MAX nodes:**  
  - $V(A_1) = \max(3, 5) = 5$, $V(A_2) = \max(6, 9) = 9$
  - $V(B_1) = \max(1, 2) = 2$, $V(B_2) = \max(0, -1) = 0$
  - $V(C_1) = \max(7, 4) = 7$, $V(C_2) = \max(5, 6) = 6$
- **Mid-level MIN nodes:**  
  - $V(A) = \min(5, 9) = 5$
  - $V(B) = \min(2, 0) = 0$
  - $V(C) = \min(7, 6) = 6$
- **Root MAX node:**  
  - $V(R) = \max(V(A), V(B), V(C)) = \max(5, 0, 6) = 6$.
- **Optimal Move for MAX:** Action **C** because it yields the highest guaranteed utility of **6** compared to 5 from A and 0 from B.

In [1]:
GAME_TREE = {
    'R': ['A', 'B', 'C'],
    'A': ['A1', 'A2'],
    'B': ['B1', 'B2'],
    'C': ['C1', 'C2'],
    'A1': ['A1a', 'A1b'],
    'A2': ['A2a', 'A2b'],
    'B1': ['B1a', 'B1b'],
    'B2': ['B2a', 'B2b'],
    'C1': ['C1a', 'C1b'],
    'C2': ['C2a', 'C2b'],
}

UTILITY = {
    'A1a': 3, 'A1b': 5,
    'A2a': 6, 'A2b': 9,
    'B1a': 1, 'B1b': 2,
    'B2a': 0, 'B2b': -1,
    'C1a': 7, 'C1b': 4,
    'C2a': 5, 'C2b': 6,
}


def minimax(node, maximizing_player=True):
    """Return the minimax value of node.

    Terminal utilities are stored in UTILITY. For non-terminal nodes,
    recursively alternate MAX and MIN according to maximizing_player.
    """
    if node in UTILITY:
        return UTILITY[node]
    
    children = GAME_TREE[node]
    if maximizing_player:
        return max(minimax(child, maximizing_player=False) for child in children)
    else:
        return min(minimax(child, maximizing_player=True) for child in children)


def minimax_decision(root='R'):
    """Return (best_child, value) for MAX at the root.

    If root actions tie, retain the first action in GAME_TREE[root].
    """
    best_child = None
    best_value = float('-inf')
    for child in GAME_TREE[root]:
        val = minimax(child, maximizing_player=False)
        if val > best_value:
            best_value = val
            best_child = child
    return best_child, best_value


move, value = minimax_decision('R')
print('Best root move:', move)
print('Minimax value :', value)


Best root move: C
Minimax value : 6


## Part 2 — Alpha–Beta Pruning

Alpha–Beta pruning returns the same decision as Minimax while skipping branches that cannot affect the final decision.


### Task 2 — Implement and analyse Alpha–Beta pruning

| Measure | Minimax | Alpha–Beta |
|---|---:|---:|
| Optimal move | C | C |
| Root value | 6 | 6 |
| Terminal leaves evaluated | 12 | 9 |
| Branches pruned | 0 | 2 |

**Evaluated Leaves (9):** `A1a`, `A1b`, `A2a`, `B1a`, `B1b`, `C1a`, `C1b`, `C2a`, `C2b`  
**Pruned Leaves / Branches (2 Cutoffs, 3 Leaves skipped):**  
1. At node $A_2$, after evaluating `A2a = 6`, $\alpha$ for $A$ becomes at least 6. But MIN node $A$ already had $\beta = 5$ from $A_1$. Since $\alpha \ge \beta$ ($6 \ge 5$), the branch to `A2b` is pruned.  
2. At node $B$, after exploring $B_1$ (which returns 2), root MAX already holds guaranteed $\alpha = 5$ from subtree $A$. Since MIN node $B$ can guarantee at most 2 (or lower), $B$ can never exceed 2. Evaluating `B2` is cut off immediately before expanding its children `B2a` and `B2b`.

**Why pruning preserves optimal decisions:**  
Pruning removes branches only when it is provable that the current player or the opponent would never choose them (because an already discovered alternative guarantees a strictly better outcome). Hence, mathematical optimality is 100% preserved.

In [2]:
import math


def alpha_beta(node, alpha=-math.inf, beta=math.inf,
               maximizing_player=True, stats=None):
    if stats is None:
        stats = {'leaves': 0, 'prunes': 0}

    if node in UTILITY:
        stats['leaves'] += 1
        return UTILITY[node], stats

    children = GAME_TREE[node]

    if maximizing_player:
        value = -math.inf
        for child in children:
            child_value, stats = alpha_beta(
                child, alpha, beta, maximizing_player=False, stats=stats
            )
            value = max(value, child_value)
            alpha = max(alpha, value)
            if alpha >= beta:
                stats['prunes'] += 1
                break
        return value, stats

    value = math.inf
    for child in children:
        child_value, stats = alpha_beta(
            child, alpha, beta, maximizing_player=True, stats=stats
        )
        value = min(value, child_value)
        beta = min(beta, value)
        if alpha >= beta:
            stats['prunes'] += 1
            break
    return value, stats


def alpha_beta_decision(root='R'):
    """Return (best_child, value, stats) for MAX at the root."""
    stats = {'leaves': 0, 'prunes': 0}
    best_child = None
    best_val = -math.inf
    alpha = -math.inf
    beta = math.inf
    
    for child in GAME_TREE[root]:
        child_val, stats = alpha_beta(child, alpha, beta, maximizing_player=False, stats=stats)
        if child_val > best_val:
            best_val = child_val
            best_child = child
        alpha = max(alpha, best_val)
    return best_child, best_val, stats


move, value, stats = alpha_beta_decision('R')
print('Best move      :', move)
print('Value          :', value)
print('Leaves examined:', stats['leaves'])
print('Prunes         :', stats['prunes'])


Best move      : C
Value          : 6
Leaves examined: 9
Prunes         : 2


## Part 3 — Standard Nim


### Task 3 — Implement and analyse Standard Nim

| Starting state | Winning or losing for player to move? | Winning move(s), if any |
|---|---|---|
| `(3, 2)` | **Winning** | `(2, 2)` |
| `(1,)` | **Winning** | `()` |
| `(1, 1)` | **Losing** | None |
| `(2, 2)` | **Losing** | None |
| `(1, 2)` | **Winning** | `(1, 1)` |

**Explanation of MAX's strategy from `(3, 2)`:**  
- From `(3, 2)`, the legal successor states are:
  - Reduce pile of 3: `(2, 2)`, `(2, 1)`, `(2,)`
  - Reduce pile of 2: `(3, 1)`, `(3,)`
- By Nim-sum / Bouton's Theorem: $3 \oplus 2 = 1 \ne 0$ (a winning position). The winning move must transition to a state with XOR sum 0.
- Reducing the pile of 3 to 2 yields `(2, 2)`, which has Nim-sum $2 \oplus 2 = 0$ (a losing state for MIN). Any move MIN makes from `(2, 2)` allows MAX to mirror and take the final piece.
- Removing 1 object from pile of 3 produces `(2, 2)` — this **is** the unique optimal winning move. Removing 2 objects gives `(2, 1)` ($2 \oplus 1 = 3 \ne 0$), which allows MIN to counter and win.

In [3]:
from functools import lru_cache


def nim_successors(state):
    """Return all distinct legal next states for standard Nim.

    A move removes a positive number of objects from exactly one pile.
    Represent each state as a tuple sorted in non-increasing order;
    omit zero-sized piles.
    """
    successors = set()
    for i, pile in enumerate(state):
        for take in range(1, pile + 1):
            rem = pile - take
            new_piles = list(state[:i] + state[i+1:])
            if rem > 0:
                new_piles.append(rem)
            successors.add(tuple(sorted(new_piles, reverse=True)))
    return sorted(list(successors), reverse=True)


@lru_cache(maxsize=None)
def nim_is_winning(state):
    """Return True if the current player has a winning strategy.

    A state with no legal moves is losing.
    """
    if not state:
        return False
    return any(not nim_is_winning(succ) for succ in nim_successors(state))


def nim_best_moves(state):
    """Return all legal moves that leave a losing state for the opponent."""
    return [succ for succ in nim_successors(state) if not nim_is_winning(succ)]


initial_nim_state = (3, 2)
print('Initial state:', initial_nim_state)
print('Winning for player to move:', nim_is_winning(initial_nim_state))
print('Winning moves:', nim_best_moves(initial_nim_state))


Initial state: (3, 2)
Winning for player to move: True
Winning moves: [(2, 2)]


## Part 4 — Grundy's Game


### Task 4 — Implement and analyse Grundy’s Game

| State | Legal successor states | Winning or losing for player to move? |
|---|---|---|
| `(5,)` | `[(4, 1), (3, 2)]` | **Winning** |
| `(4, 1)` | `[(3, 1, 1)]` | **Losing** |
| `(3, 2)` | `[(2, 2, 1)]` | **Winning** |
| `(3, 1, 1)` | `[(2, 1, 1, 1)]` | **Winning** |
| `(2, 2, 1)` | `[]` (Terminal) | **Losing** |

*(Note: In sorted representation, `(1, 4)` is represented as `(4, 1)`, and `(2, 3)` is represented as `(3, 2)`)*

**Explanation & Analysis:**
1. **Winning First Move for MAX from `(5,)`:**  
   MAX must choose **`(4, 1)`** (splitting $5 \to 4 + 1$).  
   - If MAX moves to `(4, 1)`, MIN faces a losing state because MIN's only legal response is splitting 4 into $3 + 1$, leaving `(3, 1, 1)`. From `(3, 1, 1)`, MAX splits 3 into $2 + 1$, resulting in `(2, 1, 1, 1)`. In `(2, 1, 1, 1)`, neither 2 nor 1 can be split into unequal positive integers, so MIN is left with no moves and loses!
   - If MAX mistakenly chose `(3, 2)`, MIN would split 3 into $2 + 1$, reaching `(2, 2, 1)`. In `(2, 2, 1)`, MAX has no moves and would lose.
2. **Why splitting a pile of size 2 into `(1, 1)` is illegal:**  
   The game definition strictly requires splits to produce two **unequal** pile sizes ($a \ne b$). Splitting 2 yields $1 + 1$ ($a = b = 1$), which violates this condition.
3. **Grundy's Game vs. Standard Nim:**  
   - In Standard Nim, objects are discarded from an existing pile; the total number of objects strictly decreases while pile count never increases.
   - In Grundy's Game, no objects are discarded (conservation of total objects); instead, the number of distinct piles increments on every move until all piles are of size $\le 2$ or cannot be split unequally.

In [4]:
@lru_cache(maxsize=None)
def grundy_successors(state):
    """Return all distinct legal states after one Grundy's Game move.

    Select one pile n and replace it by a+b where:
      - a + b = n
      - a >= 1 and b >= 1
      - a != b

    Return each resulting state as a tuple sorted in non-increasing order.
    """
    successors = set()
    for i, pile in enumerate(state):
        for a in range(1, (pile // 2) + 1):
            b = pile - a
            if a != b:
                new_piles = list(state[:i] + state[i+1:]) + [a, b]
                successors.add(tuple(sorted(new_piles, reverse=True)))
    return sorted(list(successors), reverse=True)


@lru_cache(maxsize=None)
def grundy_is_winning(state):
    """Return True if the current player has a winning strategy.

    A state with no legal moves is losing.
    """
    succs = grundy_successors(state)
    if not succs:
        return False
    return any(not grundy_is_winning(succ) for succ in succs)


def grundy_best_moves(state):
    """Return all legal next states that leave a losing state for the opponent."""
    return [succ for succ in grundy_successors(state) if not grundy_is_winning(succ)]


initial_grundy_state = (5,)
print('Initial state:', initial_grundy_state)
print('Legal first moves:', grundy_successors(initial_grundy_state))
print('Winning for player to move:', grundy_is_winning(initial_grundy_state))
print('Winning moves:', grundy_best_moves(initial_grundy_state))


Initial state: (5,)
Legal first moves: [(4, 1), (3, 2)]
Winning for player to move: True
Winning moves: [(4, 1)]
